# YieldGuard — wafer defect CNN

Runs end to end on a free Colab T4 in about 15 minutes. **Runtime -> Change runtime type -> T4 GPU** first.

Five cells:

1. install + Kaggle credentials
2. download and preprocess WM-811K
3. the model
4. train
5. evaluate (with and without test-time augmentation) and download

Nothing to upload. No Drive. If a cell fails, it fails loudly with the reason.

In [ ]:
#@title 1. Install + Kaggle credentials
!pip -q install kagglehub

import os
from getpass import getpass

# Token comes from kaggle.com -> Settings -> API -> Create New Token (kaggle.json).
if not os.environ.get("KAGGLE_KEY"):
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
        print("using Colab Secrets")
    except Exception:
        os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ").strip()
        os.environ["KAGGLE_KEY"]      = getpass("Kaggle key: ").strip()

print("credentials set for:", os.environ.get("KAGGLE_USERNAME"))

In [ ]:
#@title 2. Download + preprocess  (~6 min the first time, then cached)
import numpy as np, os, sys, pickle, warnings
warnings.filterwarnings("ignore")

CLASSES = ["Center","Donut","Edge-Loc","Edge-Ring","Local","Random","Scratch","Near-full","None"]
NC = 9
CACHE = "wm811k_64.npz"

if not os.path.exists(CACHE):
    import kagglehub, pandas as pd
    from PIL import Image
    from sklearn.model_selection import train_test_split

    pkl = os.path.join(kagglehub.dataset_download("qingyi/wm811k-wafer-map"), "LSWMD.pkl")
    print("downloaded:", pkl)

    # LSWMD.pkl was pickled by pandas <0.20 on Python 2 — needs two shims to load.
    import pandas.core.indexes as _ci
    sys.modules.setdefault("pandas.indexes", _ci)
    sys.modules.setdefault("pandas.indexes.base", _ci.base)
    sys.modules.setdefault("pandas.indexes.numeric", getattr(_ci, "numeric", _ci.base))
    with open(pkl, "rb") as f:
        df = pickle.load(f, encoding="latin1")
    print("records:", len(df))

    # labels live in a nested list: [[]] unlabeled, [['Center']] labeled
    fix = {"none":"None","center":"Center","donut":"Donut","edge-loc":"Edge-Loc",
           "edge-ring":"Edge-Ring","loc":"Local","local":"Local","random":"Random",
           "scratch":"Scratch","near-full":"Near-full","nearfull":"Near-full"}
    def label_of(ft):
        if isinstance(ft, np.ndarray): ft = ft.tolist()
        if not ft or not ft[0]: return None
        s = str(ft[0][0] if isinstance(ft[0], list) else ft[0]).strip()
        return fix.get(s.lower(), s)

    df["label"] = [label_of(v) for v in df["failureType"]]
    lab = df[df["label"].notna()]
    # keep 25% of "None" — it is 77% of labelled data and would swamp everything else
    none_m = lab["label"] == "None"
    use = pd.concat([lab[~none_m], lab[none_m].sample(frac=0.25, random_state=42)],
                    ignore_index=True)
    print("after None-subsampling:", len(use))

    X, y = [], []
    for wm, lb in zip(use["waferMap"], use["label"]):
        if not isinstance(wm, np.ndarray) or wm.size == 0: continue
        # NEAREST: pixels are categories (0 not-tested, 1 pass, 2 fail), not intensities
        X.append(np.array(Image.fromarray(wm.astype(np.uint8), "L")
                          .resize((64, 64), Image.NEAREST), dtype=np.uint8))
        y.append(CLASSES.index(lb))

    X = np.stack(X)[:, None]; y = np.array(y, dtype=np.int64)
    Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
    np.savez_compressed(CACHE, X_train=Xtr, y_train=ytr, X_val=Xva, y_val=yva)
    print("cached:", CACHE)

d = np.load(CACHE)
X_train = d["X_train"].astype(np.float32) / 2.0   # {0,1,2} -> {0.0,0.5,1.0}
X_val   = d["X_val"].astype(np.float32)   / 2.0
y_train, y_val = d["y_train"].astype(np.int64), d["y_val"].astype(np.int64)

print("train", X_train.shape, " val", X_val.shape)
for c, n in zip(CLASSES, np.bincount(y_val, minlength=NC)):
    print(f"  {c:<10} {n:>5}")

In [ ]:
#@title 3. Model — WaferCNN, must match src/models/vision/model.py exactly
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("!! no GPU — Runtime > Change runtime type > T4 GPU, or this takes hours")

class ResBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = nn.Sequential()
        if stride != 1 or cin != cout:
            self.shortcut = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False),
                                          nn.BatchNorm2d(cout))
    def forward(self, x):
        h = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(h)) + self.shortcut(x))

class WaferCNN(nn.Module):
    # Layer widths, strides AND attribute names are load-bearing: the repo loads this
    # checkpoint with strict=True, so any rename breaks src/models/vision/classifier.py.
    def __init__(self, nc=NC, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1, bias=False),
                                  nn.BatchNorm2d(16), nn.ReLU(inplace=True))
        self.layer1 = ResBlock(16,  32,  stride=2)   # -> 32x32
        self.layer2 = ResBlock(32,  64,  stride=2)   # -> 16x16
        self.layer3 = ResBlock(64,  128, stride=2)   # ->  8x8
        self.layer4 = ResBlock(128, 128, stride=2)   # ->  4x4
        self.gap     = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc      = nn.Linear(128, nc)
    def forward(self, x):
        x = self.layer4(self.layer3(self.layer2(self.layer1(self.stem(x)))))
        return self.fc(self.dropout(self.gap(x).flatten(1)))

n = sum(p.numel() for p in WaferCNN().parameters())
print("params:", f"{n:,}")
assert n == 615801, f"architecture drifted from the repo ({n:,} != 615,801) — the checkpoint will not load"
print("OK — matches src/models/vision/model.py")

In [ ]:
#@title 4. Train  (~8 min on a T4)
import time, copy
from sklearn.metrics import f1_score

EPOCHS = 40

class WaferData(Dataset):
    def __init__(self, X, y, aug=False):
        self.X, self.y, self.aug = torch.from_numpy(X), torch.from_numpy(y), aug
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.X[i]
        if self.aug:   # wafer patterns are dihedral-symmetric, so flips/rotations keep the label
            if torch.rand(1).item() > .5: x = x.flip(-1)
            if torch.rand(1).item() > .5: x = x.flip(-2)
            k = int(torch.randint(0, 4, (1,)).item())
            if k: x = torch.rot90(x, k, [-2, -1])
        return x, self.y[i]

# sqrt inverse-frequency: full inverse-frequency says 22 Near-full wafers deserve the same
# total gradient as 5,529 None wafers, which over-corrects hard early in training
counts = np.clip(np.bincount(y_train, minlength=NC), 1, None).astype(float)
w = (1.0 / counts) ** 0.5
w = torch.tensor(w / w.sum() * NC, dtype=torch.float, device=DEVICE)

tl = DataLoader(WaferData(X_train, y_train, aug=True), batch_size=128, shuffle=True,
                num_workers=2, pin_memory=True)
vl = DataLoader(WaferData(X_val, y_val), batch_size=512, num_workers=2)

torch.manual_seed(0)
model = WaferCNN().to(DEVICE)
crit  = nn.CrossEntropyLoss(weight=w)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

best, best_state, t0 = 0.0, None, time.time()
for ep in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in tl:
        xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    sched.step()

    model.eval(); preds = []
    with torch.no_grad():
        for xb, _ in vl: preds.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
    f1 = f1_score(y_val, np.concatenate(preds), average="macro",
                  labels=range(NC), zero_division=0)
    if f1 > best:
        best, best_state = f1, copy.deepcopy(model.state_dict())
        torch.save(best_state, "best_model.pt")
    print(f"epoch {ep:2d}/{EPOCHS}  macro-F1 {f1:.4f}   best {best:.4f}")

model.load_state_dict(best_state)
print(f"\ndone in {time.time()-t0:.0f}s — best macro-F1 {best:.4f}")

In [ ]:
#@title 5. Evaluate + download
from sklearn.metrics import accuracy_score

@torch.no_grad()
def predict(model, X, tta=False):
    model.eval()
    Xt = torch.from_numpy(X)
    total = torch.zeros(len(Xt), NC)
    views = [(k, f) for k in range(4) for f in (False, True)] if tta else [(0, False)]
    for k, flip in views:
        V = torch.rot90(Xt, k, [-2, -1])
        if flip: V = V.flip(-1)
        lg = torch.cat([model(V[i:i+512].to(DEVICE)).cpu() for i in range(0, len(V), 512)])
        total += F.softmax(lg, 1)
    return total.argmax(1).numpy()

def report(p, name):
    # labels=range(NC) matters: a class the model never predicts must score 0,
    # not silently drop out of the average and inflate the mean
    f1 = f1_score(y_val, p, average="macro", labels=range(NC), zero_division=0)
    per = f1_score(y_val, p, average=None, labels=range(NC), zero_division=0)
    never = [CLASSES[i] for i in range(NC) if (p == i).sum() == 0]
    print(f"\n{name}:  macro-F1 {f1:.4f}   accuracy {accuracy_score(y_val, p):.4f}")
    if never: print(f"  !! NEVER PREDICTS: {never}")
    return per, f1

p_base, f1_base = report(predict(model), "plain")
p_tta,  f1_tta  = report(predict(model, tta=True), "with TTA-8")

print(f"\n{'class':<12}{'support':>9}{'plain':>9}{'TTA':>9}{'delta':>9}")
print("-" * 48)
for i, c in enumerate(CLASSES):
    print(f"{c:<12}{(y_val==i).sum():>9}{p_base[i]:>9.3f}{p_tta[i]:>9.3f}{p_tta[i]-p_base[i]:>+9.3f}")
print("-" * 48)
print(f"{'MACRO-F1':<12}{'':>9}{f1_base:>9.4f}{f1_tta:>9.4f}{f1_tta-f1_base:>+9.4f}")

# Write holdout_results.json so the repo file is generated, never hand-edited.
from sklearn.metrics import precision_recall_fscore_support
import hashlib, json

pr, rc, f1s, sup = precision_recall_fscore_support(
    y_val, p_tta, labels=range(NC), zero_division=0)
pr0, rc0, f10, _ = precision_recall_fscore_support(
    y_val, p_base, labels=range(NC), zero_division=0)

json.dump({
  "model": "WaferCNN (ResNet-style)",
  "checkpoint": "checkpoints/best_model.pt",
  "checkpoint_sha256": hashlib.sha256(open("best_model.pt","rb").read()).hexdigest(),
  "params": sum(q.numel() for q in model.parameters()),
  "dataset": "WM-811K (64x64), 15% stratified held-out split, random_state=42",
  "n_val": int(len(y_val)),
  "inference": "TTA-8 (4 rotations x 2 mirrors, softmax averaged)",
  "macro_f1": round(float(f1_tta), 4),  "accuracy": round(float(accuracy_score(y_val, p_tta)), 4),
  "macro_f1_no_tta": round(float(f1_base), 4),
  "accuracy_no_tta": round(float(accuracy_score(y_val, p_base)), 4),
  "per_class": {CLASSES[i]: {"support": int(sup[i]),
                             "precision": round(float(pr[i]), 4),
                             "recall": round(float(rc[i]), 4),
                             "f1": round(float(f1s[i]), 4),
                             "f1_no_tta": round(float(f10[i]), 4)} for i in range(NC)},
  "recipe": "sqrt inverse-frequency loss weights, no sampler, CosineAnnealingLR(T_max=40), "
            "Adam lr=1e-3 wd=1e-4, 40 epochs, batch 128",
  "notes": ["macro_f1 is averaged over all 9 classes (labels=range(9)); a class the model "
            "never predicts scores 0 rather than dropping out of the average.",
            "The reported split is also the model-selection split (checkpoint = best epoch "
            "by val macro-F1), so these figures are mildly optimistic for unseen data."],
}, open("holdout_results.json","w"), indent=2)
print("wrote holdout_results.json")

from google.colab import files
files.download("best_model.pt")
files.download("holdout_results.json")

## Back in the repo

```bash
mv ~/Downloads/best_model.pt        src/models/vision/checkpoints/best_model.pt
mv ~/Downloads/holdout_results.json src/models/vision/holdout_results.json
```

Current shipped model: **macro-F1 0.9232** (0.9157 without TTA), accuracy 0.9617.

Quote macro-F1, never accuracy — `None` is 59% of the validation split, so a model that
learned nothing but "predict None" already scores 0.59 accuracy.
